'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


cell1cell23

In [ ]:
import glob
import gzip
import json
import os
import time
import pickle
import random
import re
from datetime import datetime, date
from functools import reduce
import ahocorasick  # pip install pyahocorasick
import json

# -----------------------------
# Code comment
# -----------------------------
def get_single_article_string(article):
    # Safely get title and abstract
    curr_title = article.get('title') or ''
    abstract_inverted_index = article.get('abstract_inverted_index') or {}

    # Flatten inverted index
    position_word_list = [
        (position, word)
        for word, positions in abstract_inverted_index.items()
        for position in positions
    ]

    # Assemble abstract text
    if position_word_list:
        sorted_abstract = sorted(position_word_list)
        curr_abstract = ' '.join(word for position, word in sorted_abstract)
    else:
        curr_abstract = ''

    # Clean special characters
    replace_pairs = [
        ['\n', ' '], ['-', ' '], [' \" a', 'oa'], ['\" a', 'ae'], ['\"a', 'ae'],
        [' \" o', 'oe'], ['\" o', 'oe'], ['\"o', 'oe'], [' \" u', 'ue'], ['\" u', 'ue'],
        ['\"u', 'ue'], [' \' a', 'a'], [' \' e', 'e'], [' \' o', 'o'],
        ["\' ", ""], ["\'", ""], ['  ', ' '], ['  ', ' ']
    ]

    # Concatenate title + abstract
    article_string = (curr_title + ' ' + curr_abstract).lower()
    article_string = reduce(lambda text, pair: text.replace(pair[0], pair[1]), replace_pairs, article_string)

    return article_string



def get_date_and_part_from_path(path):
    date_folder = os.path.dirname(path)
    date_str = date_folder.split('=')[-1]
    file_name = os.path.basename(path)
    part_str = file_name.split('_')[-1].split('.')[0]
    return date_str, int(part_str)


def extract_id(filename):
    match = re.search(r'log_edge_part_(\d+)_', filename)
    if match:
        return int(match.group(1))
    return None


# -----------------------------
# Code comment
# -----------------------------
log_folder = 'logs_pair'
edge_folder = "./data/concept_pair"
edge_folder_log = 'concept_pair_log'
data_folder = "data_concept_graph"
cwd = os.getcwd()
parent_dir = os.path.dirname(cwd)
concept_folder = os.path.join(parent_dir, data_folder)
progress_file = "progress.json"

for folder in [log_folder, edge_folder, edge_folder_log]:
    os.makedirs(folder, exist_ok=True)

# -----------------------------
# Code comment
# -----------------------------
base_folder = "./data/openalex/works"
file_paths = glob.glob(f'{base_folder}/updated_date=*/part_*.gz')
file_paths = sorted(file_paths, key=get_date_and_part_from_path)

start_date = datetime.strptime("2022-12-20", "%Y-%m-%d")
end_date = datetime.strptime("2025-10-01", "%Y-%m-%d")

curr_run_file_paths = [
    path for path in file_paths
    if start_date <= datetime.strptime(get_date_and_part_from_path(path)[0], "%Y-%m-%d") <= end_date
]

# -----------------------------
  # Comment
# -----------------------------
concept_folder = "./data/concept_folder"
concepts_files = os.path.join(concept_folder, 'final_concepts.txt')
with open(concepts_files, 'r', encoding='utf-8') as file:
    full_concepts = [concept.strip().lower() for concept in file.readlines() if concept.strip()]

A = ahocorasick.Automaton()
for i, concept in enumerate(full_concepts):
    A.add_word(concept, i)
A.make_automaton()

print(f"✅ Load {len(full_concepts)}  items...BuildDone...")

# -----------------------------
# Code comment
# -----------------------------
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        finished_ids = set(json.load(f))
else:
    finished_ids = set()

paper_starting_date = date(1990, 1, 1)
batch_size = 5  # Code comment
batch_save_every = 50000  # Code comment

remaining_ids = [i for i in range(len(curr_run_file_paths)) if i not in finished_ids]

# -----------------------------
# Code comment
# -----------------------------
for curr_ID in remaining_ids[:batch_size]:
    formatted_ID = '{:03d}'.format(curr_ID)
    file_path = curr_run_file_paths[curr_ID]

    edge_file = os.path.join(edge_folder, f'edge_part_{formatted_ID}.gz')
    edge_file_log = os.path.join(edge_folder_log, f'edge_part_{formatted_ID}.txt')
    log_file_txt = os.path.join(log_folder, f'log_edge_part_{formatted_ID}.txt')
    log_file_txt_finish = os.path.join(log_folder, f'log_edge_part_{formatted_ID}_finish.txt')

    current_time = datetime.now()
    with open(log_file_txt, 'a') as log_file:
        log_file.write(f'Current time: {current_time}; Start File: {file_path}\n')

    edge_lists = []
    with gzip.open(file_path, 'rt', encoding='utf-8') as file:
        lines = file.readlines()

        if not lines:
            with open(log_file_txt, 'a') as log_file:
                log_file.write("⚠️ File empty\n")
            continue

        for id_line, line in enumerate(lines):
            article_object = json.loads(line)
            get_date = datetime.strptime(article_object['publication_date'], "%Y-%m-%d").date()
            curr_paper_time = (get_date - paper_starting_date).days
            curr_all_citations = article_object.get('cited_by_count', 0)
            curr_citations_per_year = article_object.get('counts_by_year', [])
            curr_article = get_single_article_string(article_object)

            # Code comment
            concepts_for_single_paper = [idx for _, idx in A.iter(curr_article)]

            for ii in range(len(concepts_for_single_paper)):
                for jj in range(ii + 1, len(concepts_for_single_paper)):
                    edge_lists.append([
                        concepts_for_single_paper[ii],
                        concepts_for_single_paper[jj],
                        curr_paper_time,
                        curr_all_citations,
                        curr_citations_per_year
                    ])

            if len(edge_lists) >= batch_save_every:
                with gzip.open(edge_file, 'ab') as output_file:
                    pickle.dump(edge_lists, output_file)
                edge_lists = []  # Code comment

            if id_line % 5000 == 0:
                with open(log_file_txt, 'a') as log_file:
                    log_file.write(f"Processed {id_line}/{len(lines)} lines\n")

      # Comment
    if edge_lists:
        with gzip.open(edge_file, 'ab') as output_file:
            pickle.dump(edge_lists, output_file)

    with open(edge_file_log, 'a') as log_file:
        log_file.write(f'\nedge_list={len(edge_lists)}')

    with open(log_file_txt_finish, 'a') as log_file:
        log_file.write(f'Finish File: {file_path}; Time: {datetime.now()}\n')

    # UpdateProgress
    finished_ids.add(curr_ID)
    with open(progress_file, "w") as f:
        json.dump(list(finished_ids), f)

print(f"🎉 ...Done... {len(finished_ids)}  files...")


### Createedge_part_XXX.gz

In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')

      # Comment
    if os.path.exists(edge_file) or os.path.exists(finish_log):
        print(f"[{pid}] already exists...Skip")
        return

    edge_lists = []
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue

                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue

                text = get_single_article_string(article)
                if not text:
                    continue

                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue

                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])

                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        edge_lists.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])

                processed += 1

        # Save edge
        with gzip.open(edge_file, 'wb') as f:
            pickle.dump(edge_lists, f)

        # Code comment
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")

        print(f"[{pid}] Done edges={len(edge_lists)}, papers={processed}, errors={errors}")

    except Exception as e:
        print(f"[{pid}] ProcessFailed: {e}")


# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "./data/openalex/works"
    concept_file = "./data/concept_folder\\final_concepts.txt"

    edge_folder = "./data/concept_pair"
    log_folder = "./logs"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
      # Comment
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"...: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC ...BuildDone")

    # =====================================================
    # Code comment
    # =====================================================
    finished_ids = set()
    for f in glob.glob(os.path.join(log_folder, 'log_edge_part_*_finish.txt')):
        pid = extract_id(f)
        if pid is not None:
            finished_ids.add(pid)
    print(f"Completed part ...: {len(finished_ids)}")

    # =====================================================
    # Code comment
    # =====================================================
    total_tasks = 0
    for i, f in enumerate(files):
        if i in finished_ids:
            continue
        total_tasks += 1
        process_file(f, i, edge_folder, log_folder, ac_automaton)

    print(f"...: {total_tasks}")
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"All done: {datetime.now()}\n")

    print("=== ...Done ===")


In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')

      # Comment
    if os.path.exists(edge_file) or os.path.exists(finish_log):
        print(f"[{pid}] already exists...Skip")
        return

    edge_lists = []
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)

    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue

                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue

                text = get_single_article_string(article)
                if not text:
                    continue

                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue

                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])

                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        edge_lists.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])

                processed += 1

        # Save edge
        with gzip.open(edge_file, 'wb') as f:
            pickle.dump(edge_lists, f)

        # Code comment
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")

        print(f"[{pid}] Done edges={len(edge_lists)}, papers={processed}, errors={errors}")

    except Exception as e:
        print(f"[{pid}] ProcessFailed: {e}")


# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "./data/openalex/works"
    concept_file = "./data/entities.txt"

    edge_folder = "./data/entities_pair"
    log_folder = "./logs"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
      # Comment
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"...: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC ...BuildDone")

    # =====================================================
    # Code comment
    # =====================================================
    finished_ids = set()
    for f in glob.glob(os.path.join(log_folder, 'log_edge_part_*_finish.txt')):
        pid = extract_id(f)
        if pid is not None:
            finished_ids.add(pid)
    print(f"Completed part ...: {len(finished_ids)}")

    # =====================================================
    # Code comment
    # =====================================================
    total_tasks = 0
    for i, f in enumerate(files):
        if i in finished_ids:
            continue
        total_tasks += 1
        process_file(f, i, edge_folder, log_folder, ac_automaton)

    print(f"...: {total_tasks}")
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"All done: {datetime.now()}\n")

    print("=== ...Done ===")


In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

def extract_part_id_from_path(path):
    """
     [description] Extractpart...
    ...: edge_part_123.gz, temp_123, edge_part_123.pkl.gz ...
    """
    basename = os.path.basename(path)
    # Code comment
    patterns = [
        r'edge_part_(\d+)\.gz$',     # edge_part_123.gz
        r'temp_(\d+)$',              # temp_123
        
    ]
    
    for pattern in patterns:
        match = re.search(pattern, basename)
        if match:
            return int(match.group(1))
    return None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Checkpoint logic: resume if this part is completed
    if os.path.exists(finish_log) or os.path.exists(edge_file) or os.path.exists(temp_folder):
        print(f"[{pid}] already exists...edge...Done...temp...Skip")
        return
    
      # Comment
    os.makedirs(temp_folder, exist_ok=True)
    
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)
    
    # Code comment
    batch_size = 5000
    batch_count = 0
    temp_files = []
    current_batch = []
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue
                
                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue
                
                text = get_single_article_string(article)
                if not text:
                    continue
                
                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue
                
                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])
                
                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        current_batch.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])
                
                processed += 1
                
                # Code comment
                if len(current_batch) >= batch_size:
                    temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
                    with gzip.open(temp_file, 'wb') as tf:
                        pickle.dump(current_batch, tf)
                    temp_files.append(temp_file)
                    current_batch = []
                    batch_count += 1
                    print(f"[{pid}] ...Save... {batch_count}...CumulativeProcess... {processed}")

          # Comment
        if current_batch:
            temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
            with gzip.open(temp_file, 'wb') as tf:
                pickle.dump(current_batch, tf)
            temp_files.append(temp_file)
        
        # Code comment
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")
            f.write(f"Total batches: {batch_count + 1}\n")
            f.write(f"Processed papers: {processed}\n")
            f.write(f"Errors: {errors}\n")
            f.write(f"Temp folder: {temp_folder}\n")
        
        print(f"[{pid}] Done batches={batch_count + 1}, papers={processed}, errors={errors}")
        print(f"[{pid}] ...Save...: {temp_folder}")
    
    except Exception as e:
        print(f"[{pid}] ProcessFailed: {e}")
        import traceback
        traceback.print_exc()
        
          # Comment
        print(f"[{pid}] Temp fileSave...: {temp_folder}")

# =====================================================
# Code comment
# =====================================================
def get_max_completed_part(edge_folder):
    """
    ...entities_pair...Completed...part...
    1. edge_part_XXX.gz ...
    2. temp_XXX ...
    ...part...
    """
    max_part = -1
    
      # Comment
    edge_files = glob.glob(os.path.join(edge_folder, 'edge_part_*.gz'))
    for edge_file in edge_files:
        part_num = extract_part_id_from_path(edge_file)
        if part_num is not None and part_num > max_part:
            max_part = part_num
            print(f"Foundedge...: {os.path.basename(edge_file)}, part={part_num}")
    
      # Comment
    temp_folders = glob.glob(os.path.join(edge_folder, 'temp_*'))
    for temp_folder in temp_folders:
        if os.path.isdir(temp_folder):
            part_num = extract_part_id_from_path(temp_folder)
            if part_num is not None and part_num > max_part:
                max_part = part_num
                print(f"Foundtemp...: {os.path.basename(temp_folder)}, part={part_num}")
    
    return max_part

# =====================================================
# Code comment
# =====================================================
def is_part_completed(part_id, edge_folder):
    """
    Check...part...Completed
    """
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Code comment
    return os.path.exists(edge_file) or os.path.exists(temp_folder)

# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "./data/openalex/works"
    concept_file = "./data/entities.txt"

    edge_folder = "./data/entities_pair"
    log_folder = "./logs"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
      # Comment
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"...: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC ...BuildDone")

    # =====================================================
    # Code comment
    # =====================================================
    print("\n...Completed...part...")
    max_completed_part = get_max_completed_part(edge_folder)
    
    if max_completed_part >= 0:
        print(f"\n...Completed... part ...: {max_completed_part:03d}")
        print(f"... part {max_completed_part + 1:03d} StartProcess")
        
          # Comment
        skipped_parts = []
        for i in range(max_completed_part + 1):
            if is_part_completed(i, edge_folder):
                skipped_parts.append(f"{i:03d}")
        if skipped_parts:
            print(f"...Skip...Completed...parts: {', '.join(skipped_parts)}")
    else:
        print("\n...Completed... part 000 StartProcess")
        max_completed_part = -1

    # =====================================================
    # Code comment
    # =====================================================
    total_tasks = 0
    processed_count = 0
    skipped_count = 0
    
    print(f"\nTotal files: {len(files)}")
    
    for i, f in enumerate(files):
        # Code comment
        if i <= max_completed_part:
            if is_part_completed(i, edge_folder):
                skipped_count += 1
            continue
        
        total_tasks += 1
        print(f"\nStartProcess... {total_tasks}  items...: part {i:03d}")
        process_file(f, i, edge_folder, log_folder, ac_automaton)
        processed_count += 1
        
        # Code comment
        print(f"\nProgress: Completed {processed_count}/{total_tasks}  items...")

    print(f"\n{'='*50}")
    print(f"Processing completeStatistics:")
    print(f"  Total files: {len(files)}")
    print(f"  ...Skip: {skipped_count}  itemsCompleted...")
    print(f"  ...Process: {processed_count}  files")
    print(f"  ...Completed... part ...: {max_completed_part + processed_count:03d}")
    print(f"{'='*50}")
    
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"\n{'='*50}\n")
        f.write(f"All done: {datetime.now()}\n")
        f.write(f"Total files: {len(files)}\n")
        f.write(f"Skipped: {skipped_count}\n")
        f.write(f"Processed: {processed_count}\n")
        f.write(f"Last completed part: {max_completed_part + processed_count:03d}\n")
        f.write(f"{'='*50}\n")

    print("\n=== ...Done ===")

In [ ]:
import glob
import gzip
import json
import os
import pickle
import re
from datetime import datetime, date
from functools import reduce
from collections import defaultdict
from ahocorasick import Automaton

# =====================================================
# AC AUTOMATON
# =====================================================
class ACAutomaton:
    def __init__(self):
        self.goto = [{}]
        self.out = [set()]
        self.fail = [0]

    def add_word(self, word, index):
        state = 0
        for char in word:
            if char not in self.goto[state]:
                self.goto[state][char] = len(self.goto)
                self.goto.append({})
                self.out.append(set())
                self.fail.append(0)
            state = self.goto[state][char]
        self.out[state].add(index)

    def build(self):
        queue = []
        for char, state in self.goto[0].items():
            queue.append(state)
        while queue:
            r = queue.pop(0)
            for char, s in self.goto[r].items():
                queue.append(s)
                f = self.fail[r]
                while f and char not in self.goto[f]:
                    f = self.fail[f]
                self.fail[s] = self.goto[f].get(char, 0)
                self.out[s] |= self.out[self.fail[s]]

    def search(self, text):
        state = 0
        results = set()
        for char in text:
            while state and char not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(char, 0)
            if self.out[state]:
                results |= self.out[state]
        return results

# =====================================================
# HELPERS
# =====================================================
def safe_json_loads(line):
    try:
        return json.loads(line)
    except:
        return None

def safe_date_parsing(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d").date()
    except:
        return None

def get_single_article_string(article):
    try:
        title = article.get('title', '') or ''
        inv = article.get('abstract_inverted_index')
        if not inv:
            abstract = ""
        else:
            pairs = []
            for w, ps in inv.items():
                for p in ps or []:
                    pairs.append((p, w))
            abstract = ' '.join(w for _, w in sorted(pairs))

        replace_pairs = [
            ['\n', ' '], ['-', ' '], ['  ', ' ']
        ]

        text = (title + ' ' + abstract).lower()
        for a, b in replace_pairs:
            text = text.replace(a, b)

        return text
    except:
        return ""

def extract_id(filename):
    m = re.search(r'part_(\d+)', filename)
    return int(m.group(1)) if m else None

def extract_part_id_from_path(path):
    """
     [description] Extractpart...
    ...: edge_part_123.gz, temp_123, edge_part_123.pkl.gz ...
    """
    basename = os.path.basename(path)
    # Code comment
    patterns = [
        r'edge_part_(\d+)\.gz$',     # edge_part_123.gz
        r'temp_(\d+)$',              # temp_123
        
    ]
    
    for pattern in patterns:
        match = re.search(pattern, basename)
        if match:
            return int(match.group(1))
    return None

# =====================================================
# PROCESS SINGLE FILE
# =====================================================
def process_file(file_path, part_id, edge_folder, log_folder, ac_automaton):
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    finish_log = os.path.join(log_folder, f'log_edge_part_{pid}_finish.txt')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Checkpoint logic: resume if this part is completed
    if os.path.exists(finish_log) or os.path.exists(edge_file) or os.path.exists(temp_folder):
        print(f"[{pid}] already exists...edge...Done...temp...Skip")
        return
    
      # Comment
    os.makedirs(temp_folder, exist_ok=True)
    
    processed, errors = 0, 0
    paper_starting_date = date(1990, 1, 1)
    
    # Code comment
    batch_size = 5000
    max_batches = 3000  # Code comment
    batch_count = 0
    temp_files = []
    current_batch = []
    
    try:
        with gzip.open(file_path, 'rt', encoding='utf-8') as f:
            for line in f:
                article = safe_json_loads(line)
                if not article:
                    errors += 1
                    continue
                
                pub_date = safe_date_parsing(article.get('publication_date', ''))
                if not pub_date:
                    errors += 1
                    continue
                
                text = get_single_article_string(article)
                if not text:
                    continue
                
                matches = ac_automaton.search(text)
                if len(matches) < 2:
                    continue
                
                curr_time = (pub_date - paper_starting_date).days
                curr_cite = article.get('cited_by_count', 0)
                curr_cite_year = article.get('counts_by_year', [])
                
                mc = sorted(matches)
                for i in range(len(mc)):
                    for j in range(i + 1, len(mc)):
                        current_batch.append([mc[i], mc[j], curr_time, curr_cite, curr_cite_year])
                
                processed += 1
                
                # Code comment
                if len(current_batch) >= batch_size:
                    temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
                    with gzip.open(temp_file, 'wb') as tf:
                        pickle.dump(current_batch, tf)
                    temp_files.append(temp_file)
                    current_batch = []
                    batch_count += 1
                    print(f"[{pid}] ...Save... {batch_count}...CumulativeProcess... {processed}")
                    
                      # Comment
                    if batch_count >= max_batches:
                        print(f"[{pid}] ... ({max_batches})...StopProcess")
                        break

          # Comment
        if current_batch and batch_count < max_batches:
            temp_file = os.path.join(temp_folder, f'batch_{batch_count:04d}.pkl.gz')
            with gzip.open(temp_file, 'wb') as tf:
                pickle.dump(current_batch, tf)
            temp_files.append(temp_file)
            batch_count += 1
        
        # Code comment
        with open(finish_log, 'w', encoding='utf-8') as f:
            f.write(f"Finish Time: {datetime.now()}\n")
            f.write(f"Total batches: {batch_count}\n")
            f.write(f"Processed papers: {processed}\n")
            f.write(f"Errors: {errors}\n")
            f.write(f"Temp folder: {temp_folder}\n")
            if batch_count >= max_batches:
                f.write(f"Note: Reached maximum batch limit ({max_batches}), processing stopped early\n")
        
        print(f"[{pid}] Done batches={batch_count}, papers={processed}, errors={errors}")
        print(f"[{pid}] ...Save...: {temp_folder}")
    
    except Exception as e:
        print(f"[{pid}] ProcessFailed: {e}")
        import traceback
        traceback.print_exc()
        
          # Comment
        print(f"[{pid}] Temp fileSave...: {temp_folder}")

# =====================================================
# Code comment
# =====================================================
def get_max_completed_part(edge_folder):
    """
    ...entities_pair...Completed...part...
    1. edge_part_XXX.gz ...
    2. temp_XXX ...
    ...part...
    """
    max_part = -1
    
      # Comment
    edge_files = glob.glob(os.path.join(edge_folder, 'edge_part_*.gz'))
    for edge_file in edge_files:
        part_num = extract_part_id_from_path(edge_file)
        if part_num is not None and part_num > max_part:
            max_part = part_num
            print(f"Foundedge...: {os.path.basename(edge_file)}, part={part_num}")
    
      # Comment
    temp_folders = glob.glob(os.path.join(edge_folder, 'temp_*'))
    for temp_folder in temp_folders:
        if os.path.isdir(temp_folder):
            part_num = extract_part_id_from_path(temp_folder)
            if part_num is not None and part_num > max_part:
                max_part = part_num
                print(f"Foundtemp...: {os.path.basename(temp_folder)}, part={part_num}")
    
    return max_part

# =====================================================
# Code comment
# =====================================================
def is_part_completed(part_id, edge_folder):
    """
    Check...part...Completed
    """
    pid = f"{part_id:03d}"
    edge_file = os.path.join(edge_folder, f'edge_part_{pid}.gz')
    temp_folder = os.path.join(edge_folder, f'temp_{pid}')
    
    # Code comment
    return os.path.exists(edge_file) or os.path.exists(temp_folder)

# =====================================================
# ENTRY POINT
# =====================================================
if __name__ == "__main__":
    base_folder = "./data/openalex/works"
    concept_file = "./data/entities.txt"

    edge_folder = "./data/entities_pair"
    log_folder = "./logs"
    os.makedirs(edge_folder, exist_ok=True)
    os.makedirs(log_folder, exist_ok=True)

    files = sorted(glob.glob(f"{base_folder}/updated_date=*/part_*.gz"))

    # =====================================================
      # Comment
    # =====================================================
    with open(concept_file, 'r', encoding='utf-8') as f:
        full_concepts = [c.strip().lower() for c in f if c.strip()]
    print(f"...: {len(full_concepts)}")

    ac_automaton = ACAutomaton()
    for idx, concept in enumerate(full_concepts):
        ac_automaton.add_word(concept, idx)
    ac_automaton.build()
    print("AC ...BuildDone")

    # =====================================================
    # Code comment
    # =====================================================
    print("\n...Completed...part...")
    max_completed_part = get_max_completed_part(edge_folder)
    
    if max_completed_part >= 0:
        print(f"\n...Completed... part ...: {max_completed_part:03d}")
        print(f"... part {max_completed_part + 1:03d} StartProcess")
        
          # Comment
        skipped_parts = []
        for i in range(max_completed_part + 1):
            if is_part_completed(i, edge_folder):
                skipped_parts.append(f"{i:03d}")
        if skipped_parts:
            print(f"...Skip...Completed...parts: {', '.join(skipped_parts)}")
    else:
        print("\n...Completed... part 000 StartProcess")
        max_completed_part = -1

    # =====================================================
    # Code comment
    # =====================================================
    total_tasks = 0
    processed_count = 0
    skipped_count = 0
    
    print(f"\nTotal files: {len(files)}")
    
    for i, f in enumerate(files):
        # Code comment
        if i <= max_completed_part:
            if is_part_completed(i, edge_folder):
                skipped_count += 1
            continue
        
        total_tasks += 1
        print(f"\nStartProcess... {total_tasks}  items...: part {i:03d}")
        process_file(f, i, edge_folder, log_folder, ac_automaton)
        processed_count += 1
        
        # Code comment
        print(f"\nProgress: Completed {processed_count}/{total_tasks}  items...")

    print(f"\n{'='*50}")
    print(f"Processing completeStatistics:")
    print(f"  Total files: {len(files)}")
    print(f"  ...Skip: {skipped_count}  itemsCompleted...")
    print(f"  ...Process: {processed_count}  files")
    print(f"  ...Completed... part ...: {max_completed_part + processed_count:03d}")
    print(f"{'='*50}")
    
    with open("job_finish.txt", 'a', encoding='utf-8') as f:
        f.write(f"\n{'='*50}\n")
        f.write(f"All done: {datetime.now()}\n")
        f.write(f"Total files: {len(files)}\n")
        f.write(f"Skipped: {skipped_count}\n")
        f.write(f"Processed: {processed_count}\n")
        f.write(f"Last completed part: {max_completed_part + processed_count:03d}\n")
        f.write(f"{'='*50}\n")

    print("\n=== ...Done ===")

In [ ]:
! pip install pyahocorasick

In [ ]:
# Code comment
! pip install -U sentence-transformers faiss-gpu torch tqdm
# Code comment
! pip install ahocorasick

## Processing Step

Analysis and data processing code for Entity Impact Prediction project.

In [ ]:
import os
import gzip
import pickle

  # Comment
folder_path = "./data/concept_pair"

  # Comment
output_folder = os.path.join(folder_path, "filtered")
os.makedirs(output_folder, exist_ok=True)

# Code comment
gz_files = [f for f in os.listdir(folder_path) if f.endswith(".gz")]
print(f"...Found {len(gz_files)}  files...")

total_before = 0
total_after = 0

# Code comment
for gz_file in gz_files:
    file_path = os.path.join(folder_path, gz_file)
    output_path = os.path.join(output_folder, gz_file)

    try:
        with gzip.open(file_path, "rb") as f:
            edge_list = pickle.load(f)
    except Exception as e:
        print(f" ...Read {gz_file}: {e}")
        continue

    total_before += len(edge_list)

    # Code comment
    filtered = [
        edge for edge in edge_list
        if len(edge) > 3 and isinstance(edge[3], (int, float)) and edge[3] != 0
    ]

    total_after += len(filtered)

      # Comment
    print(f"\n {gz_file}")
    print(f"...: {len(edge_list)}  →  ...: {len(filtered)}")
    if filtered:
        for e in filtered[:5]:
            print(e)

          # Comment
        with gzip.open(output_path, "wb") as f:
            pickle.dump(filtered, f)
    else:
        print("...SkipSave...")

  # Comment
  # Comment
  # Comment